Resources and Inspiration

- https://cj-robinson.github.io/citi-bike-deserts/

In [ ]:
import pandas as pd
import requests
import sqlite3
from datetime import datetime



In [76]:
conn = sqlite3.connect('rideshare.db')

current = datetime.now().strftime("%Y-%m-%d-%H:00")

In [ ]:
# https://gbfs.baywheels.com/gbfs/2.3/gbfs.json Bay Wheels (SF eBikes)
# https://citibikenyc.com/system-data CitiBikes (NY)

In [ ]:
def load_json(url: str, name: str) -> pd.DataFrame:
    """
    Fetches JSON data from a given URL, extracts a specified section,
    and returns it as a pandas DataFrame.

    Args:
        url (str): URL to the JSON resource.
        name (str): Name of the key to retrieve under the language section (e.g., 'feeds').

    Returns:
        pd.DataFrame: DataFrame containing the extracted data.
    """
    sources = requests.get(url).json()
    return pd.DataFrame(sources['data'][name])


In [5]:
# read main file with links to all data sources
sources = requests.get("https://gbfs.baywheels.com/gbfs/2.3/gbfs.json").json()
sources['data']['en']['feeds']

[{'name': 'gbfs', 'url': 'https://gbfs.lyft.com/gbfs/2.3/bay/gbfs.json'},
 {'name': 'system_information',
  'url': 'https://gbfs.lyft.com/gbfs/2.3/bay/en/system_information.json'},
 {'name': 'station_information',
  'url': 'https://gbfs.lyft.com/gbfs/2.3/bay/en/station_information.json'},
 {'name': 'station_status',
  'url': 'https://gbfs.lyft.com/gbfs/2.3/bay/en/station_status.json'},
 {'name': 'free_bike_status',
  'url': 'https://gbfs.lyft.com/gbfs/2.3/bay/en/free_bike_status.json'},
 {'name': 'system_hours',
  'url': 'https://gbfs.lyft.com/gbfs/2.3/bay/en/system_hours.json'},
 {'name': 'system_calendar',
  'url': 'https://gbfs.lyft.com/gbfs/2.3/bay/en/system_calendar.json'},
 {'name': 'system_regions',
  'url': 'https://gbfs.lyft.com/gbfs/2.3/bay/en/system_regions.json'},
 {'name': 'system_pricing_plans',
  'url': 'https://gbfs.lyft.com/gbfs/2.3/bay/en/system_pricing_plans.json'},
 {'name': 'system_alerts',
  'url': 'https://gbfs.lyft.com/gbfs/2.3/bay/en/system_alerts.json'},
 {'na

### Creating Station Table

In [29]:
# stations information
st_info	 = load_json('https://gbfs.lyft.com/gbfs/2.3/bay/en/station_information.json', 'stations')
st_info.sample(3)

,rental_uris,lat,short_name,capacity,region_id,name,station_id,lon,address
26,{'android': 'https://sfo.lft.to/lastmile_qr_sc...,37.318450,SJ-Q9,15,5,Willow St at Vine St,adda58ee-59a9-4586-8210-da874168a93b,-121.883172,NaN
114,{'android': 'https://sfo.lft.to/lastmile_qr_sc...,37.868135,BK-E8,23,14,Fulton St at Bancroft Way,a4176220-2f51-47dd-82c4-342dc97867bd,-122.265937,NaN
476,{'android': 'https://sfo.lft.to/lastmile_qr_sc...,37.778799,SF-I24,31,3,San Francisco Public Library,54c91186-8493-44d1-ac4f-435350fee797,-122.415963,NaN


In [30]:
regions = load_json('https://gbfs.lyft.com/gbfs/2.3/bay/en/system_regions.json', 'regions')
regions.drop_duplicates(inplace=True)
regions.sample(3)

,name,region_id
0,San Francisco,3
1,San Jose,5
5,8D,23


In [31]:
stations = pd.merge(st_info[['station_id','name','short_name','region_id','capacity','lat','lon']],
         regions,on='region_id',how='left').dropna()
stations.rename(columns={'name_y':'region'},inplace=True)
stations

,station_id,name_x,short_name,region_id,capacity,lat,lon,region
0,bae9be55-04d4-4641-9781-3d1c4b6950f1,Saint James Park,SJ-L10,5,15,37.339301,-121.889937,San Jose
1,7d59176c-49dd-4b07-a5ab-bcc109974db3,23rd St at Taylor St,SJ-I14,5,19,37.360001,-121.878778,San Jose
2,0d730ac1-7ce6-45cf-aca6-b412ea46709d,Mission St at 1st St,SJ-H10,5,27,37.350964,-121.902016,San Jose
3,ed707a89-a68d-4921-a4cb-16c268e45a5b,San Fernando St at 7th St,SJ-M11-2,5,23,37.337122,-121.883215,San Jose
4,6994e1e6-bb3c-4309-8207-52e6004a7302,Newbury Park Dr at King Rd,SJ-H17,5,23,37.365536,-121.867966,San Jose
...,...,...,...,...,...,...,...,...
574,3c2ae472-6b36-42ce-be4d-f99208a7e8b0,Webster St at O'Farrell St,SF-H20,3,27,37.783521,-122.431158,San Francisco
575,9d4c484b-5e48-4115-8f87-e3f19b08c9d4,Jennings St at Revere Ave,SF-Y30,3,19,37.729393,-122.386537,San Francisco
576,2137006198899091688,Fruitvale Ave at MacArthur Blvd,OK-H15,12,19,37.800625,-122.216111,Oakland
577,380aa51b-4c0a-4388-ad78-37edf7338a3e,Spear St at Folsom St,SF-F30-2,3,35,37.789630,-122.390335,San Francisco


In [ ]:
# saving to database
stations.to_sql('stations', conn, if_exists='replace', index=False)

569

### Station Status Data

In [71]:
# stations status
def get_station_status(timestamp: datetime = current):
    st_status = load_json('https://gbfs.lyft.com/gbfs/2.3/bay/en/station_status.json', 'stations')
    st_status = st_status[['station_id','num_bikes_available','num_docks_available','num_ebikes_available','num_docks_disabled','num_bikes_disabled']]
    st_status['timestamp'] = timestamp
    return st_status

st_status = get_station_status()
st_status.sample(3)

,station_id,num_bikes_available,num_docks_available,num_ebikes_available,num_docks_disabled,num_bikes_disabled,timestamp
4,46b4ef45-b06b-40eb-9fdf-9bc8ff104a4f,12,3,2,0,0,2025-10-23-17:00
130,b513d838-5423-490c-bf7d-9f20771cf529,6,12,2,0,1,2025-10-23-17:00
487,df2bc9c0-0377-46c8-83ea-ef7ef9dcdb4d,14,1,2,0,0,2025-10-23-17:00


In [75]:
# create initial table
st_status.to_sql('station_status', conn, if_exists='replace', index=False)

# truncate table
cursor = conn.cursor()
try:
    cursor.execute(f"DELETE FROM station_status")
    conn.commit()
except sqlite3.Error as e:
    print(f"Error truncating table: {e}")

### Bike Status Data

In [93]:
# stations status
def get_bike_status(timestamp: datetime = current):
    bk_status = load_json('https://gbfs.lyft.com/gbfs/2.3/bay/en/free_bike_status.json', 'bikes')
    bk_status = bk_status[['bike_id','lat','lon','is_reserved','is_disabled','vehicle_type_id','current_range_meters']]
    bk_status['timestamp'] = timestamp
    return bk_status

bk_status = get_bike_status()
bk_status.sample(3)

,bike_id,lat,lon,is_reserved,is_disabled,vehicle_type_id,current_range_meters,timestamp
169,b3b3f2dd72194b21790617b907831a90,37.787709,-122.398138,0,0,2,1770.2784,2025-10-23-17:00
6,d47d0e202ab101467608b9f52ccca0e7,37.336393,-121.876875,0,0,2,16737.1776,2025-10-23-17:00
26,8b440a704cda6e8e0a9db008b9373f26,37.770833,-122.389422,0,0,2,10299.8016,2025-10-23-17:00


In [94]:
# create initial table
bk_status.to_sql('bike_status', conn, if_exists='replace', index=False)

# truncate table
cursor = conn.cursor()
try:
    cursor.execute(f"DELETE FROM bike_status")
    conn.commit()
except sqlite3.Error as e:
    print(f"Error truncating table: {e}")

In [10]:
price = requests.get("https://gbfs.lyft.com/gbfs/2.3/bay/en/system_pricing_plans.json").json()
price = price['data']['plans']
price

[{'price': '3.99',
  'is_taxable': True,
  'per_min_pricing': [{'interval': 1, 'start': 0, 'rate': 0.3}],
  'currency': 'USD',
  'plan_id': 'EBIKE_SINGLE_RIDE',
  'name': 'EBIKE SINGLE RIDE',
  'description': '$3.99 unlock fee, $0.30 per minute.'}]

In [24]:
# metrics to track

# number of bikes available
n_bikes_available = st_status['num_bikes_available'].sum()

# number of ebikes available
n_ebikes_available = st_status['num_ebikes_available'].sum()

# number of docks available
n_docks_available = st_status['num_docks_available'].sum()

# # number of bikes available per station
# n_bikes_available_per_station = st_status.groupby('station_id')['num_bikes_available'].sum()

# # number of docks available per station
# n_docks_available_per_station = st_status.groupby('station_id')['num_docks_available'].sum()

n_docks_available, n_bikes_available, n_ebikes_available, len(bk_status)

(np.int64(6799), np.int64(5641), np.int64(2884), 267)

In [20]:
stations

,station_id,name_x,short_name,region_id,capacity,lat,lon,region
0,c53990d7-f965-40f4-b305-3435e1c95a71,23rd St at Santa Clara St,SJ-M14,5,19,37.346480,-121.868570,San Jose
1,0d48fc9e-6798-46f0-bbb6-67a168800e0b,Julian St at 6th St,SJ-K11,5,15,37.342997,-121.888889,San Jose
2,ab8cc22e-0f34-4476-bf81-293cbbb2e69c,Kerley Dr at Rosemary St,SJ-F10,5,27,37.360854,-121.906834,San Jose
3,30429aed-9a47-4fd9-87fa-0db835aa8265,17th St at Santa Clara St,SJ-L13,5,19,37.343985,-121.874385,San Jose
4,bae9be55-04d4-4641-9781-3d1c4b6950f1,Saint James Park,SJ-L10,5,15,37.339301,-121.889937,San Jose
...,...,...,...,...,...,...,...,...
574,84b1ea89-19b6-4f78-92e2-2a161f75e67e,Market St at Steuart St,SF-E29-2,3,27,37.794525,-122.394880,San Francisco
575,c6fc1429-ff65-428a-920c-f07aacdf51e8,Lafayette Park,SF-E21,3,27,37.791966,-122.429315,San Francisco
576,f8b56cb7-cb70-4097-99c4-446fc29600c9,19th St at Mission St,SF-O23,3,19,37.760278,-122.419074,San Francisco
577,2029409329559634886,Sansome St at Sutter St,SF-F28-3,3,39,37.790548,-122.400605,San Francisco


569